# Motion & Eddy Current Correction

This is the most important preprocessing step — and the one where FSL and MRtrix3 differ most dramatically.

## The problem

Every dMRI volume is acquired with a **different gradient direction**. Two things corrupt each volume:

1. **Head motion**: the subject moves between volumes. Each volume sees a *different head position*, so comparing voxels across volumes compares different tissue.
2. **Eddy currents**: the rapidly changing gradients induce secondary magnetic fields (eddy currents) in conducting structures (the scanner bore). These distort the image in a direction- and b-value-dependent way.

Both effects look like translations + shears, but they are different:
- Motion is *rigid* (same for all b-values in a volume)
- Eddy currents are *linear* deformations, proportional to the gradient magnitude

## The tools

| | FSL eddy | MRtrix3 dwifslpreproc |
|---|---|---|
| **What it is** | The actual correction algorithm | A wrapper around FSL eddy + topup |
| **What it adds** | Outlier slice detection/replacement (`--repol`) | Automatic file preparation, gradient rotation |
| **Multi-shell** | Yes (with `--s2v_niter` for slice-to-vol) | Yes |
| **GPU** | Yes (`eddy_cuda`) | Via FSL |
| **Output** | Corrected NIfTI + rotated bvecs | Corrected .mif (gradients already rotated) |

> **Critical**: After eddy correction, bvecs must be **rotated** to match each corrected volume's orientation. FSL eddy does this and outputs a `.eddy_rotated_bvecs` file. MRtrix3 dwifslpreproc handles this automatically when writing .mif.

---

In [ ]:
import sys, subprocess
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from pathlib import Path

sys.path.insert(0, '../../scripts')
from utils import run

data_dir = Path('../../data/hcp/100307/T1w/Diffusion')
prep_dir = Path('../../data/hcp/100307/preprocessed')
prep_dir.mkdir(parents=True, exist_ok=True)

# Inputs (use denoised + Gibbs-corrected data from previous steps)
# If you skipped those steps, use data_dir / 'data.nii.gz' instead
dwi_in   = prep_dir / 'dwi_degibbs.mif'    # from 02_gibbs_removal
dwi_nii  = data_dir / 'data.nii.gz'         # fallback if .mif not found
bvals_f  = str(data_dir / 'bvals')
bvecs_f  = str(data_dir / 'bvecs')
mask     = str(data_dir / 'nodif_brain_mask.nii.gz')

# Outputs
eddy_out_base  = str(prep_dir / 'dwi_eddy')          # FSL eddy output base
mrt_eddy_out   = str(prep_dir / 'dwi_eddy.mif')      # MRtrix3 output

print('Setup complete.')

## Preparation: index and acquisition parameter files

FSL eddy needs two text files:

- **index file**: one integer per volume, indicating which row of the `acqparams` file describes that volume's acquisition.
- **acqparams file**: rows of `[phase-encode-dir, 0, 0, readout-time]`

For HCP data (all volumes acquired with the same PE direction):

In [ ]:
bvals = np.loadtxt(bvals_f)
n_vols = len(bvals)

# All volumes: same PE direction (AP = [0 -1 0]), same readout time
# HCP readout time ≈ 0.0646 s  (from HCP protocol)
acqparams_file = prep_dir / 'acqparams.txt'
acqparams_file.write_text('0 -1 0 0.0646\n')

# Index: all 1s (all volumes use row 1 of acqparams)
index_file = prep_dir / 'index.txt'
index_file.write_text(' '.join(['1'] * n_vols) + '\n')

print(f'acqparams : {acqparams_file.read_text().strip()}')
print(f'index     : (all {n_vols} volumes → row 1)')
print()
print('Note: If you have reverse-PE (PA) volumes, add them to acqparams')
print('and adjust the index file accordingly.')

## Approach A: FSL eddy (direct)

In [ ]:
# ─── [FSL] eddy ──────────────────────────────────────────────────────────────
#
# Key flags:
#   --repol    : replace outlier slices (signal dropout) — STRONGLY recommended
#   --cnr_maps : save contrast-to-noise ratio maps (useful for QC)
#   --residuals: save residuals (useful for diagnosing problems)
#   --niter=5  : 5 iterations (default; increase to 8 for better accuracy)
#
# For GPU: replace 'eddy_openmp' with 'eddy_cuda'
#
# Runtime: ~20-60 min on CPU, ~5 min on GPU for HCP data

fsl_eddy_cmd = [
    'eddy_openmp',              # or 'eddy_cuda' for GPU
    '--imain=' + str(dwi_nii if not dwi_in.exists() else dwi_in),
    '--mask='  + mask,
    '--index=' + str(index_file),
    '--acqp='  + str(acqparams_file),
    '--bvecs=' + bvecs_f,
    '--bvals=' + bvals_f,
    '--out='   + eddy_out_base,
    '--repol',                  # outlier replacement
    '--cnr_maps',               # QC maps
    '--niter=5',
]

print('FSL eddy command:')
print(' '.join(fsl_eddy_cmd))
print()
print('>> Uncomment the run() call below to execute (takes 20-60 min on CPU)')

# result = subprocess.run(fsl_eddy_cmd, capture_output=True, text=True)
# print(result.stdout[-2000:])  # last 2000 chars of output
# if result.returncode != 0:
#     print('STDERR:', result.stderr[-1000:])

## Approach B: MRtrix3 dwifslpreproc (recommended wrapper)

`dwifslpreproc` calls FSL `eddy` internally but handles:
- File format conversion automatically
- Gradient rotation (bvec update) embedded in the .mif output
- Temporary directory management
- Optional integration with topup for EPI distortion correction

In [ ]:
# ─── [MRtrix3] dwifslpreproc ─────────────────────────────────────────────────
#
# -rpe_none    : no reverse-PE volume available (HCP minimally preprocessed
#                data has already had topup applied)
# -pe_dir AP   : phase-encode direction
# -eddy_options: pass extra flags directly to FSL eddy
#
# If you have a reverse-PE (PA) b0 image, use:
#   -rpe_pair -se_epi <pa_b0.mif> -pe_dir AP

mrt_eddy_cmd = [
    'dwifslpreproc',
    str(dwi_in),
    mrt_eddy_out,
    '-rpe_none',
    '-pe_dir', 'AP',
    '-eddy_options', ' --repol --cnr_maps --niter=5',
    '-force',
    '-nthreads', '4',
]

print('MRtrix3 dwifslpreproc command:')
print(' '.join(mrt_eddy_cmd))
print()
print('>> Uncomment to execute:')
# result = subprocess.run(mrt_eddy_cmd, capture_output=True, text=True)
# print(result.stdout[-2000:])

## Inspecting eddy QC output

FSL eddy writes several QC files. The most important are:
- `*.eddy_parameters`: translation + rotation for each volume
- `*.eddy_outlier_report`: which slices were detected as outliers
- `*.eddy_cnr_maps`: contrast-to-noise ratio per shell

In [ ]:
# ─── QC: Motion parameters ───────────────────────────────────────────────────
params_file = Path(eddy_out_base + '.eddy_parameters')

if params_file.exists():
    params = np.loadtxt(params_file)   # shape: (n_vols, 6) — x,y,z,rx,ry,rz

    fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    axes[0].plot(params[:, :3])          # translations
    axes[0].set_ylabel('Translation (mm)')
    axes[0].legend(['x', 'y', 'z'])
    axes[0].set_title('Head Motion: Translations')

    axes[1].plot(np.rad2deg(params[:, 3:]))  # rotations in degrees
    axes[1].set_ylabel('Rotation (°)')
    axes[1].set_xlabel('Volume')
    axes[1].legend(['rx', 'ry', 'rz'])
    axes[1].set_title('Head Motion: Rotations')

    plt.tight_layout()
    plt.show()

    abs_motion = np.sqrt(np.sum(params[:, :3]**2, axis=1))
    print(f'Mean displacement : {abs_motion.mean():.2f} mm')
    print(f'Max  displacement : {abs_motion.max():.2f} mm')
    print('(>3 mm mean or >5 mm max is considered high motion)')
else:
    print('Run FSL eddy first (or use pre-run HCP data) to see motion parameters.')
    print('The HCP "minimally preprocessed" dataset has already been eddy-corrected.')

In [ ]:
# ─── QC: Outlier slices ───────────────────────────────────────────────────────
outlier_file = Path(eddy_out_base + '.eddy_outlier_report')

if outlier_file.exists():
    outliers = outlier_file.read_text()
    print('Outlier report (first 20 lines):')
    for line in outliers.split('\n')[:20]:
        print(' ', line)
else:
    print('Outlier report not found — run FSL eddy with --repol to generate it.')
    print()
    print('Typical HCP data: ~2-5% outlier slices')
    print('High-motion or clinical data: can be >20%')

## Critical point: always use the rotated bvecs!

After eddy correction, each volume has been rotated slightly (to correct for head motion). The original bvecs no longer point in the right direction for each volume. **Using the old bvecs will silently corrupt your diffusion model.**

In [ ]:
# Show how much the bvecs changed after rotation
original_bvecs_file = bvecs_f
rotated_bvecs_file  = eddy_out_base + '.eddy_rotated_bvecs'

if Path(rotated_bvecs_file).exists():
    orig = np.loadtxt(original_bvecs_file)    # (3, N)
    rot  = np.loadtxt(rotated_bvecs_file)     # (3, N)

    # Angular difference per volume (in degrees)
    dot = np.clip(np.sum(orig * rot, axis=0), -1, 1)
    angle_diff = np.rad2deg(np.arccos(np.abs(dot)))

    print(f'Max bvec rotation  : {angle_diff.max():.2f}°')
    print(f'Mean bvec rotation : {angle_diff.mean():.2f}°')
    print()
    print('Even 1° of angular error can introduce measurable FA bias.')
    print('→ Always load the .eddy_rotated_bvecs for all downstream steps.')
else:
    print('rotated_bvecs not found — this file is created by FSL eddy.')
    print('MRtrix3 dwifslpreproc embeds the rotated gradients inside the .mif file')
    print('automatically, so you never need to worry about this separately.')

---

## Summary: FSL eddy vs MRtrix3 dwifslpreproc

| | FSL eddy | MRtrix3 dwifslpreproc |
|---|---|---|
| Core algorithm | Same (FSL) | Same (FSL, called internally) |
| Gradient rotation | Manual (`.eddy_rotated_bvecs`) | Automatic (embedded in .mif) |
| File management | You handle | Automatic |
| topup integration | Separate step | Integrated with `-rpe_pair` |
| Transparency | Full control | More abstracted |
| Recommendation | Fine for scripted pipelines | Preferred for interactive use |

**Next**: [Bias field correction →](04_bias_correction.ipynb)